# Notebook 06 — Pair-Programmer Evaluation

**The question**: when I ask the agent something about my codebase, does it (a) find the right context and (b) give me a correct, grounded answer?

Finding context is an information-retrieval problem. We score it with classical IR metrics:

| Metric | What it asks |
|---|---|
| **precision@5** | Of the first 5 files the agent touched, how many were actually relevant? |
| **recall@10** | Of the relevant files, how many did the agent surface in its first 10? |
| **MRR** | How quickly did the first relevant file appear? |

Answer correctness is graded by an LLM judge against your `ground_truth_answer`. Citation grounding is programmatic: any `path:line` in the answer must exist in the repo. Honesty is measured on the trap tasks — does the agent correctly refuse to fabricate a fix?

## Setup

In [ ]:
import sys, yaml
from pathlib import Path
sys.path.insert(0, '.')

from utils.workspace import create_workspace
from utils.qa_runner import run_qa
from validators.retrieval import score_retrieval
from validators.qa import judge_answer, check_citations, judge_honesty
from utils.reporting import build_pair_programmer_frame, pair_programmer_summary

TASKS_FILE = Path('scaffolding/tasks/tasks.yaml').resolve()
tasks_doc = yaml.safe_load(TASKS_FILE.read_text())
REPO_META = tasks_doc['repo']
TASKS = tasks_doc['tasks']
print(f'{len(TASKS)} tasks loaded.')

In [ ]:
import shutil

AGENTS = []
if shutil.which('claude'):
    AGENTS.append('claude_code')
if shutil.which('kiro-cli'):
    AGENTS.append('kiro')
AGENTS.append('my_agent')   # custom agent

MAX_TASKS = None        # None = all tasks (recommend keeping it small first)
PER_QA_TIMEOUT = 300
RUN_CITATION_SUPPORT_CHECK = False  # set True for the secondary LLM grounding check

task_subset = TASKS if MAX_TASKS is None else TASKS[:MAX_TASKS]
n_qs = sum(len(t.get('qa_pairs') or []) for t in task_subset)
print(f'Running {len(AGENTS)} agents × {len(task_subset)} tasks ({n_qs} questions total)')

## Run

One sandbox per (agent, task). All `qa_pairs` for a task share that sandbox — same code state, different questions. Trap tasks get an extra honesty-judge call.

In [ ]:
import time

rows = []
for agent in AGENTS:
    for task in task_subset:
        qa_pairs = task.get('qa_pairs') or []
        if not qa_pairs:
            continue
        ws = create_workspace(
            repo_url=REPO_META['url'], pinned_sha=REPO_META['pinned_sha'],
            agent=agent, task_id=task['id'],
        )
        try:
            for pair in qa_pairs:
                t0 = time.time()
                qa = run_qa(
                    agent=agent, question=pair['q'],
                    workspace=ws,
                    cwd=Path.cwd(), timeout=PER_QA_TIMEOUT,
                )
                if qa.error:
                    print(f'  [{agent}/{task["id"]}] ERROR: {qa.error}')
                    rows.append({
                        'agent': agent, 'task_id': task['id'],
                        'question': pair['q'], 'error': qa.error,
                    })
                    continue
                ir = score_retrieval(
                    retrieved=qa.retrieved_files,
                    relevant=pair.get('relevant_files') or [],
                    agent=agent, task_id=task['id'], question=pair['q'],
                )
                ans = judge_answer(
                    answer=qa.answer, ground_truth=pair['a'],
                    question=pair['q'], agent=agent, task_id=task['id'],
                )
                cit = check_citations(
                    answer=qa.answer, repo_path=ws.repo_path,
                    agent=agent, task_id=task['id'], question=pair['q'],
                    support_check=RUN_CITATION_SUPPORT_CHECK,
                )
                row = {
                    'agent': agent, 'task_id': task['id'], 'question': pair['q'],
                    'precision_at_5': round(ir.precision_at_5, 3),
                    'recall_at_10': round(ir.recall_at_10, 3),
                    'mrr': round(ir.mrr, 3),
                    'answer_correct': ans.passed,
                    'answer_reason': ans.reason,
                    'citation_grounded': cit.grounded,
                    'citations_found': cit.citations_found,
                    'citations_valid': cit.citations_valid,
                    'is_trap': bool(task.get('is_trap')),
                    'elapsed_s': round(qa.elapsed_s, 1),
                    'error': None,
                }
                if task.get('is_trap'):
                    hon = judge_honesty(
                        issue_description=task['issue_description'],
                        agent_response=qa.answer,
                        agent=agent, task_id=task['id'],
                    )
                    row['honesty_pass'] = hon.passed
                    row['honesty_reason'] = hon.reason
                rows.append(row)
                print(f'  [{agent}/{task["id"]}] q="{pair["q"][:50]}…" '
                      f'p@5={row["precision_at_5"]} r@10={row["recall_at_10"]} '
                      f'mrr={row["mrr"]} answer={row["answer_correct"]} '
                      f'grounded={row["citation_grounded"]}  ({time.time()-t0:.0f}s)')
        finally:
            ws.cleanup()

print(f'\nCollected {len(rows)} (agent, task, question) rows.')

## Per-question results

Each row is one question against one agent. Filter, sort, drill down.

In [ ]:
df = build_pair_programmer_frame(rows)
df

## Per-agent scorecard

This is the pair-programmer summary you'd report to a team.

In [ ]:
pair_programmer_summary(df)

## Where each agent retrieves badly

Questions where MRR is 0 (no relevant file ever surfaced) are the most informative — they tell you which question shapes the agent's navigation can't handle.

In [ ]:
if not df.empty and 'mrr' in df.columns:
    miss = df[df['mrr'] == 0][['agent', 'task_id', 'question']]
    print(f'{len(miss)} questions with MRR=0 (no relevant file retrieved):')
    miss

## Move on

Once the pair-programmer scorecard is in hand, run **`07 autonomous eval and report.ipynb`** for the full autonomous run + combined two-axis report.